# EDA of parents in anatomy dataset

In [260]:
# ruff: noqa: T201, T203
from __future__ import annotations

import os
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir("..")

import logging

from dotenv import load_dotenv

from config.config import DATA_DIR
from src.evaluate import analyze_results, get_predictions_with_gt, plot_usage_histograms
from src.formatting import (
    format_oracle_pairs_filepath,
    format_run_path,
    format_storing_pathes_from_run_path,
)
from src.LLM_servers.openai import OpenAIServer
from src.onto_access import OntologyAccess
from src.onto_object import OntologyEntryAttr
from src.processing import parallel_samples_process
from src.prompts.system import BASELINE_INITIALIZATION_MESSAGE
from src.result_formatting import print_results_entry
from src.utils import read_oracle_pairs, save_run_results

import pandas as pd

logging.getLogger().setLevel(logging.WARNING)
load_dotenv()

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
llm_oracle = OpenAIServer()
llm_oracle.add_system_context(BASELINE_INITIALIZATION_MESSAGE)

In [261]:
DATASET_NAME = "anatomy"
SET_NAME = "human-mouse"
SOURCE_ONTOLOGY = "mouse"
TARGET_ONTOLOGY = "human"

src_onto_path = DATA_DIR / DATASET_NAME / SET_NAME / f"{SOURCE_ONTOLOGY}.owl"
tgt_onto_path = DATA_DIR / DATASET_NAME / SET_NAME / f"{TARGET_ONTOLOGY}.owl"

onto_src = OntologyAccess(src_onto_path, annotate_on_init=True)
onto_tgt = OntologyAccess(tgt_onto_path, annotate_on_init=True)

In [269]:
candidate_pairs_lines = pd.DataFrame(read_oracle_pairs(format_oracle_pairs_filepath(DATASET_NAME, SET_NAME)))

In [270]:
candidate_pairs_lines.head()

,0,1
0,http://human.owl#NCI_C32658,http://mouse.owl#MA_0002684
1,http://human.owl#NCI_C33075,http://mouse.owl#MA_0001249
2,http://human.owl#NCI_C33748,http://mouse.owl#MA_0001221
3,http://human.owl#NCI_C12262,http://mouse.owl#MA_0001615
4,http://human.owl#NCI_C53050,http://mouse.owl#MA_0002234


In [271]:
duplicated1col = candidate_pairs_lines[0][candidate_pairs_lines[0].duplicated()]
duplicated2col = candidate_pairs_lines[1][candidate_pairs_lines[1].duplicated()]

In [272]:
print(f"Number unique vals col 1: {len(set(candidate_pairs_lines[0]))}")
print(duplicated1col)

Number unique vals col 1: 373
28     http://human.owl#NCI_C12442
65     http://human.owl#NCI_C49482
72     http://human.owl#NCI_C49210
79     http://human.owl#NCI_C33661
88     http://human.owl#NCI_C32154
101    http://human.owl#NCI_C13004
105    http://human.owl#NCI_C32933
116    http://human.owl#NCI_C25763
133    http://human.owl#NCI_C33894
165    http://human.owl#NCI_C12421
178    http://human.owl#NCI_C12807
183    http://human.owl#NCI_C32934
186    http://human.owl#NCI_C12759
205    http://human.owl#NCI_C48257
213    http://human.owl#NCI_C49479
229    http://human.owl#NCI_C32931
251    http://human.owl#NCI_C33581
269    http://human.owl#NCI_C48940
294    http://human.owl#NCI_C33417
296    http://human.owl#NCI_C12386
316    http://human.owl#NCI_C32573
318    http://human.owl#NCI_C32824
349    http://human.owl#NCI_C49577
360    http://human.owl#NCI_C53050
391    http://human.owl#NCI_C12470
Name: 0, dtype: object


In [273]:
print(f"Num unique vals col 2: {len(set(candidate_pairs_lines[1]))}")
print(duplicated2col)

Num unique vals col 2: 378
91     http://mouse.owl#MA_0001771
178    http://mouse.owl#MA_0001029
190    http://mouse.owl#MA_0001289
195    http://mouse.owl#MA_0001596
225    http://mouse.owl#MA_0000279
237    http://mouse.owl#MA_0001036
264    http://mouse.owl#MA_0001630
269    http://mouse.owl#MA_0001633
274    http://mouse.owl#MA_0000050
284    http://mouse.owl#MA_0002283
285    http://mouse.owl#MA_0002684
288    http://mouse.owl#MA_0002284
294    http://mouse.owl#MA_0001669
312    http://mouse.owl#MA_0002672
322    http://mouse.owl#MA_0000854
329    http://mouse.owl#MA_0001651
371    http://mouse.owl#MA_0001012
377    http://mouse.owl#MA_0000288
391    http://mouse.owl#MA_0000014
395    http://mouse.owl#MA_0000165
Name: 1, dtype: object


### Restructure source - target ontologies

In [274]:
target_cols = candidate_pairs_lines[0]
source_cols = candidate_pairs_lines[1]

In [275]:
def get_df(cols: list[str], onto: OntologyAccess) -> pd.DataFrame:
    data = [
        {
            "column_name": col,
            "all_entity_names": OntologyEntryAttr(col, onto).get_all_entity_names(),
            "parents_preferred_names": OntologyEntryAttr(col, onto).get_parents_preferred_names(),
            "direct_parents": OntologyEntryAttr(col, onto).get_direct_parents(),
        }
        for col in cols
    ]
    return pd.DataFrame(data)

tgt_df = get_df(target_cols, onto_tgt)
src_df = get_df(source_cols, onto_src)

In [276]:
# code clean
src_df["parents_preferred_names"] = src_df["parents_preferred_names"].apply(lambda x: [eval(i) for i in str(x)[1:-1].split(",") if 'owl.Thing' not in i])
tgt_df["parents_preferred_names"] = tgt_df["parents_preferred_names"].apply(lambda x: [eval(i) for i in str(x)[1:-1].split(",") if 'owl.Thing' not in i])

src_df["direct_parents"] = src_df["direct_parents"].apply(lambda x: str(x))
tgt_df["direct_parents"] = tgt_df["direct_parents"].apply(lambda x: str(x))

In [277]:
src_df.head()

,column_name,all_entity_names,parents_preferred_names,direct_parents
0,http://mouse.owl#MA_0002684,{stomach muscularis mucosa},[{Thing}],{owl.Thing}
1,http://mouse.owl#MA_0001249,"{palpebral gland, meibomian gland, tarsal gland}","[{eye gland}, {Thing}]",{mouse.MA_0000267}
2,http://mouse.owl#MA_0001221,{tensor tympani},"[{tympanic cavity muscle}, {Thing}]",{mouse.MA_0000256}
3,http://mouse.owl#MA_0001615,{stomach greater curvature},"[{Thing}, {stomach region}]",{mouse.MA_0002561}
4,http://mouse.owl#MA_0002234,{testicular vein},"[{Thing}, {venous blood vessel}, {blood vessel...",{mouse.MA_0000067}


In [278]:
tgt_df.head()

,column_name,all_entity_names,parents_preferred_names,direct_parents
0,http://human.owl#NCI_C32658,"{Gastric Muscularis Mucosa, Gastric_Muscularis...","[{Other_Anatomic_Concept}, {Gastric_Tissue}, {...","{human.NCI_C32656, human.NCI_C33149}"
1,http://human.owl#NCI_C33075,"{Meibomian_Gland, Meibomian Gland}","[{Anatomic_Structure_System_or_Substance}, {Gl...",{human.NCI_C33519}
2,http://human.owl#NCI_C33748,"{Tensor_Tympani, Tensor Tympani}","[{Body_Part}, {Head_and_Neck_Muscle}, {Muscle_...",{human.NCI_C33148}
3,http://human.owl#NCI_C12262,"{Greater Curvature of the Stomach, Greater_Cur...","[{Thing}, {Anatomic_Structure_System_or_Substa...",{human.NCI_C25763}
4,http://human.owl#NCI_C53050,"{Testicular Vein, Internal Spermatic Vein, Spe...","[{Blood_Vessel}, {Body_Part}, {Anatomic_Struct...",{human.NCI_C12814}


In [ ]:
# display_scrollable(tgt_df)

### Select best direct entry

In [279]:
def select_best_direct_entity(src_entity: OntologyEntryAttr, tgt_entity: OntologyEntryAttr) -> list[OntologyEntryAttr, OntologyEntryAttr]:
    """
    If there are multiple direct parents, select one and find child element for it
    """
    src_parents = list(src_entity.get_direct_parents())[0]
    tgt_parents = list(tgt_entity.get_direct_parents())[0]

    src_children = list(src_parents.get_direct_children())[0]
    tgt_children = list(tgt_parents.get_direct_children())[0]
    
    return [src_parents, tgt_parents, src_children, tgt_children]

<b><i> Example ontology</i></b>:

<b> Number </b>: 315

<b> Column name </b>: http://mouse.owl#MA_0000293

<b> All entity names </b>: {'pelvic girdle bone'}

<b> Parents preffered names </b>: "[{'pelvis bone'}, {'trunk bone'}, {'bone'}, {'abdomen/pelvis/perineum bone'}, {'girdle bone'}, {owl.Thing}]"

<b> Direct parents </b>: "{mouse.MA_0000291, mouse.MA_0000532}"

In [250]:
mouse_example = OntologyEntryAttr("http://mouse.owl#MA_0000293", onto_src)

In [256]:
parent = list(mouse_example.get_direct_parents())[0]
parent

mouse.MA_0000291

In [258]:
child = list(parent.get_direct_children())[0]
child

mouse.MA_0000293

### Format hierarchy

In [ ]:
def format_hierarchy(hierarchy_dict):
    """
    >>> entry_uri = "http://human.owl#NCI_C33075"
    >>> entry = OntologyEntryAttr(entry_uri, onto_tgt)
    >>> format_hierarchy(entry.get_parents_by_levels(max_level=8), onto_tgt)
    Level 0: Meibomian_Gland
    Level 1: Sebaceous Gland
    Level 2: A gland from which secretions reach a free surface of the body by ducts.
    Level 3: gland
    Level 4: Organ
    Level 5: Anatomic Structures and Systems
    Level 6: Thing
    """
    formatted = []
    for level, parents in sorted(hierarchy_dict.items()):
        parent_name =  list(list(parents)[0].get_all_entity_names())[0]
        formatted.append(f"    Level {level}: {parent_name}")
    return "\n".join(formatted)

### Prompt templates

In [ ]:
def prompt_direct_entity(src_entity: OntologyEntryAttr, tgt_entity: OntologyEntryAttr) -> str:
    """Regular prompt that uses natural language and is more intuitive."""
    src_parent, tgt_parent, src_child, tgt_child = select_best_direct_entity(src_entity, tgt_entity)
    return f"""
    Here are two biomedical entities from different ontologies. Each is shown with its name and its direct parent.

    1. Source entity: "{src_child}"  
       - Parent: "{src_parent}"

    2. Target entity: "{tgt_child}"  
       - Parent: "{tgt_parent}"

    Do these entities mean the same thing? Respond with "True" or "False".
    """


def prompt_direct_entity_ontological(src_entity: OntologyEntryAttr, tgt_entity: OntologyEntryAttr) -> str:
    """Ontological prompt that uses ontology-focused language, emphasizing terms like ontological concept, hierarchical placement, semantic equivalence, etc."""
    src_parent, tgt_parent, src_child, tgt_child = select_best_direct_entity(src_entity, tgt_entity)
    return f"""
    Analyze the following entities, each originating from a distinct biomedical ontology. Your task is to assess whether they represent the **same ontological concept**, considering both their semantic meaning and hierarchical position.

    1. Source entity: "{src_child}"  
       - Immediate ontological parent: "{src_parent}"

    2. Target entity: "{tgt_child}"  
       - Immediate ontological parent: "{tgt_parent}"

    Are these entities **ontologically equivalent** within their respective ontologies? Respond with "True" or "False".
    """


def prompt_sequential_hierarchy(src_entity: OntologyEntryAttr, tgt_entity: OntologyEntryAttr) -> str:
    """Regular prompt that uses natural language and is more intuitive."""
    src_hierarchy = format_hierarchy(src_entity.get_parents_by_levels(max_level=8))
    tgt_hierarchy = format_hierarchy(tgt_entity.get_parents_by_levels(max_level=8))

    return f"""
    Below are two biomedical entities, each described with its **full hierarchy** from its respective ontology.

    1. Source entity hierarchy:  
    {src_hierarchy}

    2. Target entity hierarchy:  
    {tgt_hierarchy}

    Do these entities represent the **same biomedical concept**? Respond with "True" or "False".
    """


def prompt_sequential_hierarchy_ontological(src_entity: OntologyEntryAttr, tgt_entity: OntologyEntryAttr) -> str:
    """Ontological prompt that uses ontology-focused language, emphasizing terms like ontological concept, hierarchical placement, semantic equivalence, etc."""
    src_hierarchy = format_hierarchy(src_entity.get_parents_by_levels(max_level=8))
    tgt_hierarchy = format_hierarchy(tgt_entity.get_parents_by_levels(max_level=8))

    return f"""
    The following entities come from distinct biomedical ontologies. Each is represented by its **ontological lineage**, capturing its hierarchical placement from the most general to the most specific level.

    1. Source entity ontological lineage:  
    {src_hierarchy}

    2. Target entity ontological lineage:  
    {tgt_hierarchy}

    Based on their **ontological positioning, hierarchical relationships, and semantic alignment**, do these entities represent the **same ontological concept**? Respond with "True" or "False".
    """

### Run experiments

In [ ]:
MAX_WORKERS = 6
MODEL = "gpt-4o-mini"

EXP_NAMES_TO_PROMPT_FUNCTIONS = {
    "direct_entity": prompt_direct_entity,
    "direct_entity_ontological": prompt_direct_entity_ontological,
    "sequential_hierarchy": prompt_sequential_hierarchy,
    "sequential_hierarchy_ontological": prompt_sequential_hierarchy_ontological,
}


oracle_candidate_pairs_path = format_oracle_pairs_filepath(DATASET_NAME, SET_NAME)

oracle_candidate_pairs = read_oracle_pairs(oracle_candidate_pairs_path)[:10]

run_path = format_run_path()

for current_exp_name, prompt_function in EXP_NAMES_TO_PROMPT_FUNCTIONS.items():
    results, tokens_usage, confidences = parallel_samples_process(
        oracle_candidate_pairs, llm_oracle, onto_src, onto_tgt, MODEL, MAX_WORKERS, prompt_function
    )
    prediction_path, stats_path, diagram_path = format_storing_pathes_from_run_path(
        run_path, SET_NAME, MODEL, current_exp_name
    )
    save_run_results(results, prediction_path, columns=["Source", "Target", "Prediction", "Confidence"])

    plot_usage_histograms(tokens_usage, confidences, do_plot=True, do_print=True, suptitle=current_exp_name)

    predictions = get_predictions_with_gt(run_path, DATASET_NAME, SET_NAME, MODEL, current_exp_name)
    analyze_results(
        predictions,
        print_results=True,
        plot_confusion_matrix=True,
        subtitle=f"{SET_NAME}: {MODEL} {current_exp_name} | ",
        cm_save_path=diagram_path,
        stats_path=stats_path,
    )
    print("\n\n\n")
    break
